In [12]:
import warnings
warnings.filterwarnings("ignore")
 
import pandas as pd
from sklearn.model_selection import train_test_split, StratifiedKFold, GridSearchCV
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.feature_selection import SelectKBest, f_classif
from xgboost import XGBClassifier
from sklearn.metrics import (accuracy_score, precision_score, recall_score,
                             f1_score, roc_auc_score)
        
 
df = pd.read_csv("alzheimers_disease_data.csv")
X = df.drop(columns=["Diagnosis", "PatientID", "DoctorInCharge"], errors="ignore")
y = df["Diagnosis"]
 
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=SEED)
 
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
 
 
def make_xgb():
    return XGBClassifier(n_estimators=200, learning_rate=0.1, max_depth=2,
                         random_state=SEED, n_jobs=-1, eval_metric="logloss")
 
 
def evaluate(name, model, n_features):
    pred = model.predict(X_test)
    proba = model.predict_proba(X_test)[:, 1]
    return {"Setup": name, "#Features": n_features,
            "Accuracy": accuracy_score(y_test, pred),
            "Precision": precision_score(y_test, pred),
            "Recall": recall_score(y_test, pred),
            "F1": f1_score(y_test, pred),
            "ROC AUC": roc_auc_score(y_test, proba)}
 
 
results = []
 
# 1) BEFORE feature selection, no scaling (baseline)
m = make_xgb().fit(X_train, y_train)
results.append(evaluate("Before FS | no scaling", m, X.shape[1]))
 
# 2) BEFORE feature selection, scaled (ใช้ทุก feature)
m = Pipeline([("scale", StandardScaler()), ("clf", make_xgb())]).fit(X_train, y_train)
results.append(evaluate("Before FS | scaled", m, X.shape[1]))
 
# 3) AFTER feature selection, scaled
#    scaler -> SelectKBest -> XGBoost ใน pipeline เดียว: fit จาก train set เท่านั้น
#    และเลือกจำนวน feature (k) ด้วย CV จึงไม่มี data leakage
pipe = Pipeline([("scale", StandardScaler()),
                 ("select", SelectKBest(f_classif)),
                 ("clf", make_xgb())])
gs = GridSearchCV(pipe, {"select__k": [5, 10, 15, 20, 25]},
                  cv=cv, scoring="roc_auc", n_jobs=-1).fit(X_train, y_train)
best = gs.best_estimator_
kept = list(X.columns[best.named_steps["select"].get_support()])
results.append(evaluate(f"After FS | scaled (k={gs.best_params_['select__k']})",
                        best, len(kept)))
 
# 4) AFTER feature selection, no scaling (เทียบว่า scaling มีผลไหม)
pipe2 = Pipeline([("select", SelectKBest(f_classif)), ("clf", make_xgb())])
gs2 = GridSearchCV(pipe2, {"select__k": [5, 10, 15, 20, 25]},
                   cv=cv, scoring="roc_auc", n_jobs=-1).fit(X_train, y_train)
results.append(evaluate(f"After FS | no scaling (k={gs2.best_params_['select__k']})",
                        gs2.best_estimator_, gs2.best_params_["select__k"]))
 
res = pd.DataFrame(results).set_index("Setup")
print(res.round(4).to_string())
print("\nSelected features (After FS | scaled):")
print(kept)
res.round(4).to_csv("xgboost_scaled_fs_results.csv")

                              #Features  Accuracy  Precision  Recall      F1  ROC AUC
Setup                                                                                
Before FS | no scaling               32    0.9512     0.9456  0.9145  0.9298   0.9428
Before FS | scaled                   32    0.9512     0.9456  0.9145  0.9298   0.9428
After FS | scaled (k=20)             20    0.9512     0.9456  0.9145  0.9298   0.9445
After FS | no scaling (k=20)         20    0.9512     0.9456  0.9145  0.9298   0.9445

Selected features (After FS | scaled):
['Age', 'EducationLevel', 'BMI', 'SleepQuality', 'FamilyHistoryAlzheimers', 'CardiovascularDisease', 'Diabetes', 'HeadInjury', 'Hypertension', 'SystolicBP', 'CholesterolLDL', 'CholesterolHDL', 'CholesterolTriglycerides', 'MMSE', 'FunctionalAssessment', 'MemoryComplaints', 'BehavioralProblems', 'ADL', 'Disorientation', 'PersonalityChanges']
